===============================================================================STEP 4 of 6 IN THE FULL PIPELINE - epoch_change_stats.py===============================================================================PURPOSE: Computes per-epoch EPR and full-period LRR vegetation-edge changerates for every transect in the merged corridor (Step 3's output), using theFULL dense archive of detections - not reduced to annual snapshots. Thisproduces Table 4.1 and the per-transect veg_retreat_rate used in the CVI.THREE LAYERS OF PROTECTION AGAINST NOISY/IMPLAUSIBLE RATES:  1. Point-level filtering: implausible single date-distance jumps are     dropped from the raw series before fitting (robust median-deviation     check), so one bad detection doesn't distort a whole transect's rate.  2. Robust regression: Theil-Sen (median-of-slopes), not ordinary least     squares - far less sensitive to any outliers that survive step 1.  3. Structure-aware epoching: transects crossing a hard structure (here,     Lekki Deep Sea Port) built partway through the record are no longer     tracking natural shoreline change from that point on - they're     tracking a hard structure edge, or nothing at all. Any epoch/rate     that would represent post-construction "movement" for these     transects is NOT reported as a natural shoreline-change rate; it is     excluded and flagged instead, and reported separately (see Table     labelled 'Lekki Deep Sea Port' below), so it can never silently     pollute the natural, site-wide summary statistics.Run this AFTER Step 3. Also run Step 5 (waterline_change_stats.py) - theyare independent scripts reading different pickles, not one script doingboth.Run with: (coastguard) $ python epoch_change_stats.py

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

In [ ]:
# %% Header banner + imports

print("=" * 70)
print("RUNNING: epoch_change_stats.py - THEIL-SEN + HARBOR-AWARE VERSION")
print("(point-level MAD filtering + Theil-Sen regression + per-epoch mean+/-2SD")
print(" cutoff + Lekki Deep Sea Port transects excluded from natural rates)")
print("=" * 70)
print()

import pickle
import numpy as np
import pandas as pd
from scipy.stats import theilslopes

In [ ]:
# %% Load merged transect-intersections data, define epochs

sitename = 'LEKKI'   # the MERGED_SITE name from Step 3 - the validated 13-segment corridor
with open(f'Data/{sitename}/intersections/{sitename}_transect_intersects.pkl', 'rb') as f:
    TransectInterGDF = pickle.load(f)

epochs = [('2013-01-01', '2015-12-31'),
          ('2015-01-01', '2020-12-31'),
          ('2020-01-01', '2025-12-31'),
          ('2013-01-01', '2025-12-31')]
epoch_labels = ['2013-2015', '2015-2020', '2020-2025', '2013-2025']

In [ ]:
# %% Outlier / rate thresholds

POINT_JUMP_THRESHOLD_M = 60   # a single date-to-date jump bigger than this (m) is
                               # treated as an implausible detection, not real change
POINT_Z_THRESHOLD = 6

# Hard physical ceiling on epoch RATES (not raw points): applied BEFORE the
# mean+/-2SD pass, so a noisy epoch can't let an implausible rate hide
# inside a wide distribution.
RATE_HARD_CUTOFF_M_YR = 15

In [ ]:
# %% Hard-structure configuration (Lekki Deep Sea Port)

# --- Hard-structure configuration ------------------------------------------
# Lekki Deep Sea Port - construction start confirmed absent in 2013/2015
# imagery, present from 2020. If you ever revisit the eastward extension,
# add a second dict entry here for the Dangote Refinery Jetty (confirmed:
# transects 1037-1040 in the extended 19-segment merge, construction 2017) -
# but that structure is not reachable in this validated 13-segment corridor,
# so it's intentionally omitted here.
STRUCTURES = [
    {
        "name": "Lekki Deep Sea Port",
        "transects": set(range(728, 745)),   # 728 to 744 inclusive - confirmed via
                                              # source_segment sanity check below
        "date": pd.Timestamp('2020-01-01'),
    },
]

_TRANSECT_TO_STRUCTURE = {}
for _struct in STRUCTURES:
    for _tid in _struct["transects"]:
        _TRANSECT_TO_STRUCTURE[_tid] = _struct


def structure_for(tid):
    return _TRANSECT_TO_STRUCTURE.get(tid)

In [ ]:
# %% Sanity check against STRUCTURES

# --- Sanity check before trusting STRUCTURES above -------------------------
# A structure's TransectID set was identified against a PAST merge run.
# Since Step 3 fully re-derives TransectID from scratch every run (0..N-1,
# in ANALYSIS_SEGMENT_SITES list order), it's only still correct if segment
# order/inclusion hasn't changed since. This prints which segment(s) the
# defined structure's transects actually landed in - expect ONE segment
# name; if you see more than one, or an unrecognised segment, STOP and
# re-derive the transect set before trusting anything downstream.
if 'source_segment' in TransectInterGDF.columns:
    for struct in STRUCTURES:
        check = TransectInterGDF[TransectInterGDF['TransectID'].isin(struct["transects"])]
        print(f"Sanity check - segment(s) '{struct['name']}' transects actually fall in:")
        print(check['source_segment'].value_counts())
        print("(Expect ONE segment name here.)\n")
else:
    print("WARNING: 'source_segment' column not found - cannot verify STRUCTURES "
          "still points at the right place.\n")

In [ ]:
# %% Helper functions: point-outlier filtering, epoch window, epoch rate

def filter_point_outliers(dates_sorted, dists_sorted):
    """Drop individual points whose distance value jumps implausibly far from
    the local median - catches single bad detections without discarding the
    whole transect/epoch."""
    if len(dists_sorted) < 3:
        return dates_sorted, dists_sorted
    dists = np.array(dists_sorted, dtype=float)
    med = np.median(dists)
    mad = np.median(np.abs(dists - med)) or 1.0
    z = np.abs(dists - med) / (1.4826 * mad)
    keep = z < POINT_Z_THRESHOLD
    return [d for d, k in zip(dates_sorted, keep) if k], dists[keep]


def resolve_epoch_window(tid, start, end):
    """An epoch is only usable for a structure-affected transect if it ends
    entirely before that structure's date. Truncating to a partial
    pre-construction window was tried previously and consistently failed
    (too few clean points to fit), so it isn't attempted."""
    struct = structure_for(tid)
    if struct is None:
        return start, end, 'natural'
    if end < struct["date"]:
        return start, end, 'natural'
    return start, end, 'structure_excluded'


def epoch_rate(tid, dates, distances, start, end):
    eval_start, eval_end, flag = resolve_epoch_window(tid, start, end)
    if flag == 'structure_excluded':
        return np.nan, flag

    dates = pd.to_datetime(dates)
    mask = (dates >= eval_start) & (dates <= eval_end)
    if mask.sum() < 3:
        return np.nan, flag

    d_sub = dates[mask]
    y_sub = np.array(distances)[mask]
    order = np.argsort(d_sub.values)
    d_sorted = d_sub.values[order]
    y_sorted = y_sub[order]

    d_clean, y_clean = filter_point_outliers(list(d_sorted), y_sorted)
    if len(y_clean) < 3:
        return np.nan, flag

    t_years = (pd.to_datetime(d_clean) - pd.to_datetime(d_clean).min()).days / 365.25
    if t_years.max() - t_years.min() < 0.5:
        return np.nan, flag

    slope, intercept, lo_slope, hi_slope = theilslopes(y_clean, t_years)
    return -slope, flag   # negated so that negative = landward retreat

In [ ]:
# %% Compute per-transect, per-epoch rates (main loop)

records = []
for _, row in TransectInterGDF.iterrows():
    tid = row['TransectID']
    dates = row['dates']
    dists = row['distances']
    struct = structure_for(tid)
    struct_name = struct["name"] if struct is not None else None
    for (start, end), label in zip(epochs, epoch_labels):
        rate, flag = epoch_rate(tid, dates, dists, pd.Timestamp(start), pd.Timestamp(end))
        records.append({'TransectID': tid, 'epoch': label, 'rate_m_yr': rate,
                         'flag': flag, 'structure': struct_name})

FullRateDF = pd.DataFrame(records)
FullRateDF.to_csv(f'Data/{sitename}/vegedge_change_per_transect_ALL_including_excluded.csv', index=False)

In [ ]:
# %% Report structure-affected transect epoch handling

print("Structure-affected transect epoch handling:")
for struct in STRUCTURES:
    struct_rows = FullRateDF[FullRateDF['structure'] == struct["name"]]
    print(f"  '{struct['name']}':")
    for label in epoch_labels:
        epoch_rows = struct_rows[struct_rows['epoch'] == label]
        for flag, n in epoch_rows['flag'].value_counts().items():
            print(f"    {label}: {n} transects -> '{flag}'")
print()

In [ ]:
# %% Clean rates: hard cutoff + mean +/- 2SD

RateDF = FullRateDF.dropna(subset=['rate_m_yr']).copy()

n_before_hard = len(RateDF)
hard_outliers = RateDF[RateDF['rate_m_yr'].abs() > RATE_HARD_CUTOFF_M_YR]
if len(hard_outliers):
    print(f"Hard cutoff: removing {len(hard_outliers)} of {n_before_hard} rates with "
          f"|rate| > {RATE_HARD_CUTOFF_M_YR} m/yr (implausible, not a real cutoff-fit).")
RateDF = RateDF[RateDF['rate_m_yr'].abs() <= RATE_HARD_CUTOFF_M_YR].copy()

n_before = len(RateDF)
cleaned_parts = []
RateDF['structure_group'] = RateDF['structure'].fillna('natural')
for epoch, group in RateDF.groupby('epoch'):
    for group_name, subgroup in group.groupby('structure_group'):
        if len(subgroup) < 2:
            cleaned_parts.append(subgroup)
            continue
        mu, sigma = subgroup['rate_m_yr'].mean(), subgroup['rate_m_yr'].std()
        lo, hi = mu - 2 * sigma, mu + 2 * sigma
        outliers = subgroup[(subgroup['rate_m_yr'] < lo) | (subgroup['rate_m_yr'] > hi)]
        if len(outliers):
            print(f"{epoch} ({group_name}): removing {len(outliers)} of {len(subgroup)} rates "
                  f"outside mean +/- 2SD = [{lo:.2f}, {hi:.2f}] m/yr")
        cleaned_parts.append(subgroup[(subgroup['rate_m_yr'] >= lo) & (subgroup['rate_m_yr'] <= hi)])
RateDF = pd.concat(cleaned_parts, ignore_index=True)
print(f"\n{len(RateDF)} of {n_before_hard} epoch-transect rates retained overall "
      f"(after hard cutoff + mean+/-2SD).\n")

In [ ]:
# %% Classify status, build Table 4.1 (natural transects)

def classify(rate):
    if rate < -0.5:
        return 'retreating'
    elif rate > 0.5:
        return 'advancing'
    return 'stable'


RateDF['status'] = RateDF['rate_m_yr'].apply(classify)

NaturalRateDF = RateDF[RateDF['structure'].isna()]
summary = NaturalRateDF.groupby('epoch').agg(
    mean_rate=('rate_m_yr', 'mean'), min_rate=('rate_m_yr', 'min'),
    max_rate=('rate_m_yr', 'max'), n=('rate_m_yr', 'count'))
status_pct = (NaturalRateDF.groupby(['epoch', 'status']).size()
              .unstack(fill_value=0).apply(lambda r: 100 * r / r.sum(), axis=1))
table_4_1 = summary.join(status_pct).round(2).reindex(epoch_labels)
print("Table 4.1 (natural transects only, all defined structures excluded):")
print(table_4_1)
table_4_1.to_csv(f'Data/{sitename}/vegedge_change_by_epoch_natural.csv')

In [ ]:
# %% Structure-specific summaries

for struct in STRUCTURES:
    StructRateDF = RateDF[RateDF['structure'] == struct["name"]]
    if len(StructRateDF):
        struct_summary = StructRateDF.groupby('epoch').agg(
            mean_rate=('rate_m_yr', 'mean'), min_rate=('rate_m_yr', 'min'),
            max_rate=('rate_m_yr', 'max'), n=('rate_m_yr', 'count')).reindex(epoch_labels)
        safe_name = struct["name"].lower().replace(" ", "_")
        print(f"\n'{struct['name']}' transects - report/map separately. "
              f"Only epochs fully before {struct['date'].date()} are genuine natural rates:")
        print(struct_summary)
        struct_summary.to_csv(f'Data/{sitename}/vegedge_change_by_epoch_{safe_name}.csv')

In [ ]:
# %% Save per-transect rates; build single veg_retreat_rate for CVI join

RateDF.to_csv(f'Data/{sitename}/vegedge_change_per_transect_new.csv', index=False)

# ----------------------------------------------------------------------
# Single per-transect veg_retreat_rate for the CVI join (Part B, Step 1 of
# the ArcGIS guide), per Chapter 3 Section 3.6's own rule: mean of
# available sub-period EPR values, or the full-period LRR alone where
# sub-period rates are unavailable.
# ----------------------------------------------------------------------
subperiods = ['2013-2015', '2015-2020', '2020-2025']
full_period = '2013-2025'
pivot = RateDF.pivot_table(index='TransectID', columns='epoch', values='rate_m_yr')
for col in subperiods + [full_period]:
    if col not in pivot.columns:
        pivot[col] = np.nan
sub_mean = pivot[subperiods].mean(axis=1, skipna=True)
veg_retreat_rate = sub_mean.where(sub_mean.notna(), pivot[full_period])
cvi_join = veg_retreat_rate.reset_index()
cvi_join.columns = ['TransectID', 'veg_retreat_rate']
cvi_join.to_csv(f'Data/{sitename}/veg_rate_for_cvi_join.csv', index=False)
print(f"\nSaved -> Data/{sitename}/veg_rate_for_cvi_join.csv "
      f"({cvi_join['veg_retreat_rate'].notna().sum()} of {len(cvi_join)} transects have a usable rate)")
print("Join this onto your transect layer in ArcGIS for the CVI's veg_retreat_rate field.")